In [2]:
%pip install "psycopg[binary]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.5/215.5 kB 7.7 MB/s eta 0:00:00


In [4]:
# Test the postgreSQl connection
from google.colab import userdata
import psycopg

DATABASE_URL = userdata.get("NEON_DB_URL")

with psycopg.connect(DATABASE_URL, connect_timeout=15) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT
                current_database(),
                current_user,
                version()
        """)
        result = cur.fetchone()

print("Connection successful!")
print("Database:", result[0])
print("Role:", result[1])
print("PostgreSQL:", result[2].split(",")[0])

Connection successful!
Database: neondb
Role: neondb_owner
PostgreSQL: PostgreSQL 18.6 (653b019) on aarch64-unknown-linux-gnu


# Inspect the six CSV outputs for database schema design

In [9]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import sys

# Existing GitHub repository cloned in Google Drive
REPO_ROOT = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)

# Raw and processed data directories
RAW_DIR = REPO_ROOT / "data" / "raw" / "olist"
PROCESSED_DIR = REPO_ROOT / "data" / "processed"

# Python source directory
SRC_DIR = REPO_ROOT / "src"

# Make project modules importable
sys.path.insert(0, str(SRC_DIR))

# Verify the setup
print("Repository exists:", REPO_ROOT.exists())
print("Raw data directory exists:", RAW_DIR.exists())
print("Processed data directory exists:", PROCESSED_DIR.exists())
print("Source directory exists:", SRC_DIR.exists())

Mounted at /content/drive
Repository exists: True
Raw data directory exists: True
Processed data directory exists: True
Source directory exists: True


In [10]:

from pathlib import Path
import pandas as pd

PROCESSED_DIR = REPO_ROOT / "data" / "processed"

csv_files = [
    "order_analytics.csv",
    "product_analytics.csv",
    "category_analytics.csv",
    "customer_analytics.csv",
    "monthly_sales_analytics.csv",
    "order_status_summary.csv",
]

for filename in csv_files:
    file_path = PROCESSED_DIR / filename

    if not file_path.exists():
        print(f"\nMISSING FILE: {filename}")
        continue

    df = pd.read_csv(file_path, nrows=1000)

    print(f"\n{'=' * 60}")
    print(f"FILE: {filename}")
    print(f"COLUMNS: {len(df.columns)}")
    print(f"DATA TYPES AND SAMPLE VALUES:")

    for column in df.columns:
        print(
            f"  {column}: "
            f"{df[column].dtype} | "
            f"sample={df[column].dropna().head(2).tolist()}"
        )



FILE: order_analytics.csv
COLUMNS: 21
DATA TYPES AND SAMPLE VALUES:
  order_id: object | sample=['e481f51cbdc54678b7cc49136f2d6af7', '53cdb2fc8bc7dce0b6741e2150273451']
  customer_id: object | sample=['9ef432eb6251297304e76186b10a928d', 'b0830fb4747a6c6d20dea0b8c802d7ef']
  order_status: object | sample=['delivered', 'delivered']
  order_purchase_timestamp: object | sample=['2017-10-02 10:56:33', '2018-07-24 20:41:37']
  order_approved_at: object | sample=['2017-10-02 11:07:15', '2018-07-26 03:24:27']
  order_delivered_carrier_date: object | sample=['2017-10-04 19:55:00', '2018-07-26 14:31:00']
  order_delivered_customer_date: object | sample=['2017-10-10 21:25:13', '2018-08-07 15:27:45']
  order_estimated_delivery_date: object | sample=['2017-10-18', '2018-08-13']
  item_count: int64 | sample=[1, 1]
  unique_products: int64 | sample=[1, 1]
  unique_sellers: int64 | sample=[1, 1]
  item_revenue: float64 | sample=[29.99, 118.7]
  freight_cost: float64 | sample=[8.72, 22.76]
  payment_t

PostgeSQL warehouse schema

In [11]:

from pathlib import Path

database_file = SRC_DIR / "sales_pipeline" / "database.py"

print("Target file:", database_file)
print("Parent directory exists:", database_file.parent.exists())
print("File already exists:", database_file.exists())


Target file: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/src/sales_pipeline/database.py
Parent directory exists: True
File already exists: True


In [12]:


schema_code = r'''"""Database loading functions for the sales analytics warehouse."""

import psycopg


CREATE_TABLES_SQL = """
CREATE TABLE IF NOT EXISTS order_analytics (
    order_id TEXT PRIMARY KEY,
    customer_id TEXT NOT NULL,
    order_status TEXT,
    order_purchase_timestamp TIMESTAMP,
    order_approved_at TIMESTAMP,
    order_delivered_carrier_date TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date DATE,
    item_count INTEGER,
    unique_products INTEGER,
    unique_sellers INTEGER,
    item_revenue NUMERIC(14, 2),
    freight_cost NUMERIC(14, 2),
    payment_total NUMERIC(14, 2),
    payment_records INTEGER,
    payment_types INTEGER,
    delivery_duration_days DOUBLE PRECISION,
    estimated_delivery_duration_days DOUBLE PRECISION,
    delivery_delay_days DOUBLE PRECISION,
    is_delivered BOOLEAN,
    is_late_delivery BOOLEAN
);

CREATE TABLE IF NOT EXISTS product_analytics (
    product_id TEXT PRIMARY KEY,
    product_category_name_english TEXT,
    units_sold INTEGER,
    item_revenue NUMERIC(14, 2),
    freight_cost NUMERIC(14, 2),
    distinct_orders INTEGER,
    distinct_sellers INTEGER
);

CREATE TABLE IF NOT EXISTS category_analytics (
    product_category_name_english TEXT PRIMARY KEY,
    units_sold INTEGER,
    item_revenue NUMERIC(14, 2),
    freight_cost NUMERIC(14, 2),
    distinct_orders INTEGER,
    distinct_products INTEGER,
    distinct_sellers INTEGER
);

CREATE TABLE IF NOT EXISTS customer_analytics (
    customer_unique_id TEXT PRIMARY KEY,
    order_count INTEGER,
    item_revenue NUMERIC(14, 2),
    first_order_date TIMESTAMP,
    last_order_date TIMESTAMP,
    orders_with_revenue INTEGER,
    customer_state TEXT,
    average_order_value NUMERIC(14, 2),
    is_repeat_customer BOOLEAN
);

CREATE TABLE IF NOT EXISTS monthly_sales_analytics (
    order_month TEXT PRIMARY KEY,
    order_count INTEGER,
    item_revenue NUMERIC(14, 2),
    freight_cost NUMERIC(14, 2),
    total_items DOUBLE PRECISION,
    orders_with_revenue INTEGER,
    average_order_value NUMERIC(14, 2)
);

CREATE TABLE IF NOT EXISTS order_status_summary (
    order_status TEXT PRIMARY KEY,
    order_count INTEGER
);
"""


def create_tables(database_url: str) -> None:
    """Create the analytics tables if they do not already exist."""
    with psycopg.connect(database_url) as conn:
        with conn.cursor() as cur:
            cur.execute(CREATE_TABLES_SQL)
'''

database_file.write_text(schema_code, encoding="utf-8")
print("Updated:", database_file)
print("File size:", database_file.stat().st_size, "bytes")



Updated: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/src/sales_pipeline/database.py
File size: 2424 bytes


Create tables in neon

In [13]:
# update the create_tables() function

schema_code = database_file.read_text(encoding="utf-8")

old_function = '''def create_tables(database_url: str) -> None:
    """Create the analytics tables if they do not already exist."""
    with psycopg.connect(database_url) as conn:
        with conn.cursor() as cur:
            cur.execute(CREATE_TABLES_SQL)
'''

new_function = '''def create_tables(database_url: str) -> None:
    """Create the analytics tables if they do not already exist."""
    with psycopg.connect(database_url) as conn:
        with conn.cursor() as cur:
            for statement in CREATE_TABLES_SQL.split(";"):
                if statement.strip():
                    cur.execute(statement)
'''

if old_function not in schema_code:
    raise ValueError(
        "Expected function not found. File was not modified."
    )

database_file.write_text(
    schema_code.replace(old_function, new_function),
    encoding="utf-8"
)

print("Updated create_tables() successfully.")


Updated create_tables() successfully.


In [14]:
#execute the schema

from google.colab import userdata
from sales_pipeline.database import create_tables


create_tables(DATABASE_URL)

print("Schema creation completed successfully.")

#varify the tables

import psycopg

expected_tables = {
    "order_analytics",
    "product_analytics",
    "category_analytics",
    "customer_analytics",
    "monthly_sales_analytics",
    "order_status_summary",
}

with psycopg.connect(DATABASE_URL) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT table_name
            FROM information_schema.tables
            WHERE table_schema = 'public'
              AND table_type = 'BASE TABLE'
            ORDER BY table_name;
        """)
        actual_tables = {row[0] for row in cur.fetchall()}

print("Tables found:", sorted(actual_tables))
print("Expected tables present:", expected_tables.issubset(actual_tables))
print("Missing tables:", sorted(expected_tables - actual_tables))



Schema creation completed successfully.
Tables found: ['category_analytics', 'customer_analytics', 'monthly_sales_analytics', 'order_analytics', 'order_status_summary', 'product_analytics']
Expected tables present: True
Missing tables: []


Implement the CSE-to-PostgreSQL loader

In [15]:

loader_code = r'''

import csv
from io import StringIO
from pathlib import Path

from psycopg import sql


CSV_TABLE_MAP = {
    "order_analytics.csv": "order_analytics",
    "product_analytics.csv": "product_analytics",
    "category_analytics.csv": "category_analytics",
    "customer_analytics.csv": "customer_analytics",
    "monthly_sales_analytics.csv": "monthly_sales_analytics",
    "order_status_summary.csv": "order_status_summary",
}


def load_processed_csvs(
    database_url: str,
    processed_dir: str | Path,
) -> dict[str, dict[str, int]]:
    """Atomically replace warehouse table contents from processed CSVs."""

    processed_dir = Path(processed_dir)

    # Fail before modifying the database if any file is missing.
    missing_files = [
        filename
        for filename in CSV_TABLE_MAP
        if not (processed_dir / filename).is_file()
    ]

    if missing_files:
        raise FileNotFoundError(
            f"Missing processed CSV files: {missing_files}"
        )

    results = {}

    # A single transaction covers all six tables.
    # If any load fails, all truncations and inserts roll back.
    with psycopg.connect(database_url) as conn:
        with conn.cursor() as cur:

            # Read and validate the CSV headers before truncating.
            for filename, table_name in CSV_TABLE_MAP.items():
                with (processed_dir / filename).open(
                    "r", encoding="utf-8-sig", newline=""
                ) as file:
                    reader = csv.reader(file)
                    csv_columns = next(reader, None)

                cur.execute(
                    """
                    SELECT column_name
                    FROM information_schema.columns
                    WHERE table_schema = 'public'
                      AND table_name = %s
                    ORDER BY ordinal_position
                    """,
                    (table_name,),
                )
                db_columns = [row[0] for row in cur.fetchall()]

                if csv_columns != db_columns:
                    raise ValueError(
                        f"Column mismatch for {filename}. "
                        f"CSV={csv_columns}; DB={db_columns}"
                    )

            # Full refresh: replace all six table contents.
            table_identifiers = [
                sql.Identifier(table)
                for table in CSV_TABLE_MAP.values()
            ]
            cur.execute(
                sql.SQL("TRUNCATE TABLE {}").format(
                    sql.SQL(", ").join(table_identifiers)
                )
            )

            for filename, table_name in CSV_TABLE_MAP.items():
                file_path = processed_dir / filename

                with file_path.open(
                    "r", encoding="utf-8-sig", newline=""
                ) as file:
                    csv_buffer = StringIO(file.read())

                copy_query = sql.SQL(
                    "COPY {} FROM STDIN WITH "
                    "(FORMAT CSV, HEADER TRUE, NULL '')"
                ).format(sql.Identifier(table_name))

                with cur.copy(copy_query) as copy:
                    copy.write(csv_buffer.getvalue())

                cur.execute(
                    sql.SQL("SELECT COUNT(*) FROM {}").format(
                        sql.Identifier(table_name)
                    )
                )
                loaded_rows = cur.fetchone()[0]

                # Count source records independently of the database.
                with file_path.open(
                    "r", encoding="utf-8-sig", newline=""
                ) as file:
                    source_rows = sum(1 for _ in csv.reader(file)) - 1

                if loaded_rows != source_rows:
                    raise RuntimeError(
                        f"Row-count mismatch for {table_name}: "
                        f"CSV={source_rows}, DB={loaded_rows}"
                    )

                results[table_name] = {
                    "source_rows": source_rows,
                    "loaded_rows": loaded_rows,
                }

    return results
'''

existing_code = database_file.read_text(encoding="utf-8")

if "def load_processed_csvs(" in existing_code:
    print("Loader already exists. No changes made.")
else:
    with database_file.open("a", encoding="utf-8") as file:
        file.write(loader_code)
    print("CSV loader added successfully.")


CSV loader added successfully.


In [16]:
#Run the initial data load

import importlib
import sales_pipeline.database as database

importlib.reload(database)

from google.colab import userdata

DATABASE_URL = userdata.get("NEON_DB_URL")

load_results = database.load_processed_csvs(
    database_url=DATABASE_URL,
    processed_dir=PROCESSED_DIR,
)

for table_name, counts in load_results.items():
    print(
        f"{table_name}: "
        f"source={counts['source_rows']:,}, "
        f"loaded={counts['loaded_rows']:,}"
    )


order_analytics: source=99,441, loaded=99,441
product_analytics: source=32,729, loaded=32,729
category_analytics: source=72, loaded=72
customer_analytics: source=94,990, loaded=94,990
monthly_sales_analytics: source=24, loaded=24
order_status_summary: source=8, loaded=8


# Validate the warehouse using SQL


Run the SQL validation queries

In [17]:

import psycopg
from psycopg.rows import dict_row

validation_queries = {
    "1. Order and delivery metrics": """
        SELECT
            COUNT(*) AS total_orders,
            COUNT(*) FILTER (WHERE is_delivered) AS delivered_orders,
            COUNT(*) FILTER (WHERE is_late_delivery) AS late_orders,
            ROUND(
                100.0 * COUNT(*) FILTER (WHERE is_delivered)
                / NULLIF(COUNT(*), 0), 2
            ) AS delivery_rate_pct
        FROM order_analytics;
    """,

    "2. Revenue and freight totals": """
        SELECT
            ROUND(SUM(item_revenue), 2) AS total_item_revenue,
            ROUND(SUM(freight_cost), 2) AS total_freight_cost,
            ROUND(SUM(payment_total), 2) AS total_payment_amount
        FROM order_analytics;
    """,

    "3. Monthly sales trend": """
        SELECT
            order_month,
            order_count,
            ROUND(item_revenue, 2) AS item_revenue,
            ROUND(average_order_value, 2) AS average_order_value
        FROM monthly_sales_analytics
        ORDER BY order_month;
    """,

    "4. Top 10 products by revenue": """
        SELECT
            product_id,
            product_category_name_english AS category,
            units_sold,
            ROUND(item_revenue, 2) AS item_revenue
        FROM product_analytics
        ORDER BY item_revenue DESC NULLS LAST
        LIMIT 10;
    """,

    "5. Top 10 categories by revenue": """
        SELECT
            product_category_name_english AS category,
            units_sold,
            distinct_products,
            ROUND(item_revenue, 2) AS item_revenue
        FROM category_analytics
        ORDER BY item_revenue DESC NULLS LAST
        LIMIT 10;
    """,

    "6. Customer metrics": """
        SELECT
            COUNT(*) AS total_customers,
            COUNT(*) FILTER (
                WHERE is_repeat_customer
            ) AS repeat_customers,
            ROUND(AVG(average_order_value), 2) AS avg_customer_order_value
        FROM customer_analytics;
    """,

    "7. Order status distribution": """
        SELECT
            order_status,
            order_count
        FROM order_status_summary
        ORDER BY order_count DESC;
    """,
}

with psycopg.connect(DATABASE_URL, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        for title, query in validation_queries.items():
            cur.execute(query)
            rows = cur.fetchall()

            print(f"\n{'=' * 65}")
            print(title)
            print("=" * 65)

            for row in rows:
                print(dict(row))



1. Order and delivery metrics
{'total_orders': 99441, 'delivered_orders': 96478, 'late_orders': 7826, 'delivery_rate_pct': Decimal('97.02')}

2. Revenue and freight totals
{'total_item_revenue': Decimal('13591643.70'), 'total_freight_cost': Decimal('2251909.54'), 'total_payment_amount': Decimal('16008872.12')}

3. Monthly sales trend
{'order_month': '2016-09', 'order_count': 2, 'item_revenue': Decimal('207.86'), 'average_order_value': Decimal('103.93')}
{'order_month': '2016-10', 'order_count': 293, 'item_revenue': Decimal('44507.30'), 'average_order_value': Decimal('153.47')}
{'order_month': '2016-12', 'order_count': 1, 'item_revenue': Decimal('10.90'), 'average_order_value': Decimal('10.90')}
{'order_month': '2017-01', 'order_count': 787, 'item_revenue': Decimal('120098.27'), 'average_order_value': Decimal('152.60')}
{'order_month': '2017-02', 'order_count': 1718, 'item_revenue': Decimal('244959.35'), 'average_order_value': Decimal('142.58')}
{'order_month': '2017-03', 'order_count'

In [18]:
# Check aggregate reconciliation


import psycopg

reconciliation_sql = """
SELECT
    (SELECT COUNT(*) FROM order_analytics) AS order_rows,
    (SELECT SUM(order_count) FROM monthly_sales_analytics)
        AS monthly_order_total,
    (SELECT SUM(order_count) FROM order_status_summary)
        AS status_order_total,

    (SELECT ROUND(SUM(item_revenue), 2)
     FROM order_analytics) AS order_revenue,

    (SELECT ROUND(SUM(item_revenue), 2)
     FROM monthly_sales_analytics) AS monthly_revenue,

    (SELECT COUNT(*) FROM product_analytics) AS product_rows,
    (SELECT COUNT(*) FROM category_analytics) AS category_rows,
    (SELECT COUNT(*) FROM customer_analytics) AS customer_rows;
"""

with psycopg.connect(DATABASE_URL) as conn:
    with conn.cursor() as cur:
        cur.execute(reconciliation_sql)
        row = cur.fetchone()

        columns = [desc.name for desc in cur.description]
        results = dict(zip(columns, row))

for key, value in results.items():
    print(f"{key}: {value}")

print("\n--- Reconciliation checks ---")
print("Order counts match:",
      results["order_rows"] == results["monthly_order_total"]
      == results["status_order_total"])

print("Order and monthly revenue match:",
      results["order_revenue"] == results["monthly_revenue"])


order_rows: 99441
monthly_order_total: 98207
status_order_total: 99441
order_revenue: 13591643.70
monthly_revenue: 13494400.74
product_rows: 32729
category_rows: 72
customer_rows: 94990

--- Reconciliation checks ---
Order counts match: False
Order and monthly revenue match: False


In [19]:
#Identify the missing orders and revenue

import psycopg

diagnostic_sql = """
SELECT
    COUNT(*) AS total_orders,

    COUNT(*) FILTER (
        WHERE order_purchase_timestamp IS NULL
    ) AS orders_without_purchase_timestamp,

    ROUND(
        COALESCE(
            SUM(item_revenue) FILTER (
                WHERE order_purchase_timestamp IS NULL
            ),
            0
        ),
        2
    ) AS revenue_without_purchase_timestamp,

    COUNT(*) FILTER (
        WHERE order_purchase_timestamp IS NOT NULL
    ) AS orders_with_purchase_timestamp,

    ROUND(
        COALESCE(
            SUM(item_revenue) FILTER (
                WHERE order_purchase_timestamp IS NOT NULL
            ),
            0
        ),
        2
    ) AS revenue_with_purchase_timestamp

FROM order_analytics;
"""

with psycopg.connect(DATABASE_URL) as conn:
    with conn.cursor() as cur:
        cur.execute(diagnostic_sql)
        print(dict(zip(
            [column.name for column in cur.description],
            cur.fetchone()
        )))


{'total_orders': 99441, 'orders_without_purchase_timestamp': 0, 'revenue_without_purchase_timestamp': Decimal('0.00'), 'orders_with_purchase_timestamp': 99441, 'revenue_with_purchase_timestamp': Decimal('13591643.70')}


In [22]:
#varify the excluded order status
import psycopg

query = """
SELECT
    order_status,
    COUNT(*) AS order_count,
    COALESCE(ROUND(SUM(item_revenue), 2), 0) AS item_revenue
FROM order_analytics
GROUP BY order_status
ORDER BY order_status;
"""

with psycopg.connect(DATABASE_URL) as conn:
    with conn.cursor() as cur:
        cur.execute(query)
        print(f"{'STATUS':<18} {'ORDERS':>10} {'REVENUE':>16}")
        print("-" * 46)

        for status, count, revenue in cur.fetchall():
            print(f"{status:<18} {count:>10,} {revenue:>16,.2f}")


STATUS                 ORDERS          REVENUE
----------------------------------------------
approved                    2           209.60
canceled                  625        95,235.27
created                     5             0.00
delivered              96,478    13,221,498.11
invoiced                  314        61,526.37
processing                301        60,439.22
shipped                 1,107       150,727.44
unavailable               609         2,007.69


In [27]:
# inspect pipeline

pipeline_code = pipeline_file.read_text(encoding="utf-8")

old_signature = "def run_pipeline(data_dir, output_dir):"
new_signature = "def run_pipeline(data_dir, output_dir, database_url=None):"

if old_signature not in pipeline_code:
    raise ValueError(
        "Expected function signature not found. No changes made."
    )

if "load_processed_csvs" in pipeline_code:
    print("Database integration already appears to exist. No changes made.")
else:
    pipeline_code = pipeline_code.replace(
        old_signature, new_signature, 1
    )

    import_line = "from . import transform_sales\n"
    database_import = "from . import database\n"

    if import_line not in pipeline_code:
        raise ValueError(
            "Expected import line not found. No changes made."
        )

    pipeline_code = pipeline_code.replace(
        import_line, import_line + database_import, 1
    )

    marker = '''    finished_at = datetime.now(timezone.utc)
    duration = (finished_at - started_at).total_seconds()
'''

    database_stage = '''    database_results = None

    if database_url is not None:
        print("\\\\n[6/6] Loading analytics into PostgreSQL...")
        database_results = database.load_processed_csvs(
            database_url=database_url,
            processed_dir=output_dir,
        )
        print("PostgreSQL loading completed successfully.")

    finished_at = datetime.now(timezone.utc)
    duration = (finished_at - started_at).total_seconds()
'''

    if marker not in pipeline_code:
        raise ValueError(
            "Expected summary section not found. No changes made."
        )

    pipeline_code = pipeline_code.replace(
        marker, database_stage, 1
    )

    summary_marker = '''        "outputs": [str(path) for path in outputs],
'''

    summary_addition = '''        "outputs": [str(path) for path in outputs],
        "database_loaded": database_results is not None,
        "database_results": database_results,
'''

    if summary_marker not in pipeline_code:
        raise ValueError(
            "Expected summary dictionary entry not found. No changes made."
        )

    pipeline_code = pipeline_code.replace(
        summary_marker, summary_addition, 1
    )

    pipeline_file.write_text(pipeline_code, encoding="utf-8")
    print("Updated pipeline.py successfully.")




ValueError: Expected function signature not found. No changes made.

In [28]:

pipeline_code = pipeline_file.read_text(encoding="utf-8")

old_line = r'print("\\n[6/6] Loading analytics into PostgreSQL...")'
new_line = r'print("\n[6/6] Loading analytics into PostgreSQL...")'

if old_line in pipeline_code:
    pipeline_code = pipeline_code.replace(old_line, new_line, 1)
    pipeline_file.write_text(pipeline_code, encoding="utf-8")
    print("Print statement fixed.")
else:
    print("Exact old line not found. No changes made.")


Print statement fixed.


In [29]:

import importlib
import inspect
import sales_pipeline.pipeline as pipeline

importlib.reload(pipeline)

print("Function signature:")
print(inspect.signature(pipeline.run_pipeline))

assert "database_url" in inspect.signature(
    pipeline.run_pipeline
).parameters

print("\nIntegration signature check passed.")


Function signature:
(data_dir, output_dir, database_url=None)

Integration signature check passed.


In [30]:

import importlib
import sales_pipeline.pipeline as pipeline

importlib.reload(pipeline)

summary_csv = pipeline.run_pipeline(
    data_dir=RAW_DIR,
    output_dir=PROCESSED_DIR,
)

print("\n--- CSV-only mode verification ---")
print("Database loaded:", summary_csv["database_loaded"])
print("Database results:", summary_csv["database_results"])
print("Order rows:", summary_csv["order_rows"])
print("Product rows:", summary_csv["product_rows"])
print("Customer rows:", summary_csv["customer_rows"])

assert summary_csv["database_loaded"] is False
assert summary_csv["database_results"] is None
assert summary_csv["order_rows"] == 99441
assert summary_csv["product_rows"] == 32729
assert summary_csv["customer_rows"] == 94990

print("\nCSV-only mode test passed.")


SALES ANALYTICS DATA PIPELINE

[1/5] Validating source data...
SALES ANALYTICS PIPELINE — DATA QUALITY REPORT
[PASS] Customers: customer_id has no nulls
        Missing values: 0
[PASS] Customers: customer_id is unique
        Duplicate rows for key: 0
[PASS] Orders: order_id has no nulls
        Missing values: 0
[PASS] Orders: order_id is unique
        Duplicate rows for key: 0
[PASS] Order items: order_id has no nulls
        Missing values: 0
[PASS] Order items: order_item_id has no nulls
        Missing values: 0
[PASS] Order items: order_id, order_item_id is unique
        Duplicate rows for key: 0
[PASS] Products: product_id has no nulls
        Missing values: 0
[PASS] Products: product_id is unique
        Duplicate rows for key: 0
[PASS] Sellers: seller_id has no nulls
        Missing values: 0
[PASS] Sellers: seller_id is unique
        Duplicate rows for key: 0
[PASS] Orders: customer_id has no nulls
        Missing values: 0
[PASS] Order items: product_id has no nulls
   

In [32]:
#Test the integrated PostgreSQL mode

from google.colab import userdata
import importlib
import sales_pipeline.pipeline as pipeline

importlib.reload(pipeline)

DATABASE_URL = userdata.get("NEON_DB_URL")

summary_db = pipeline.run_pipeline(
    data_dir=RAW_DIR,
    output_dir=PROCESSED_DIR,
    database_url=DATABASE_URL,
)

print("\n--- PostgreSQL integration verification ---")
print("Database loaded:", summary_db["database_loaded"])

for table_name, counts in summary_db["database_results"].items():
    print(
        f"{table_name}: "
        f"source={counts['source_rows']:,}, "
        f"loaded={counts['loaded_rows']:,}"
    )

assert summary_db["database_loaded"] is True
assert summary_db["database_results"] is not None

for table_name, counts in summary_db["database_results"].items():
    assert counts["source_rows"] == counts["loaded_rows"]

assert len(summary_db["database_results"]) == 6

print("\nPostgreSQL integration test passed.")


SALES ANALYTICS DATA PIPELINE

[1/5] Validating source data...
SALES ANALYTICS PIPELINE — DATA QUALITY REPORT
[PASS] Customers: customer_id has no nulls
        Missing values: 0
[PASS] Customers: customer_id is unique
        Duplicate rows for key: 0
[PASS] Orders: order_id has no nulls
        Missing values: 0
[PASS] Orders: order_id is unique
        Duplicate rows for key: 0
[PASS] Order items: order_id has no nulls
        Missing values: 0
[PASS] Order items: order_item_id has no nulls
        Missing values: 0
[PASS] Order items: order_id, order_item_id is unique
        Duplicate rows for key: 0
[PASS] Products: product_id has no nulls
        Missing values: 0
[PASS] Products: product_id is unique
        Duplicate rows for key: 0
[PASS] Sellers: seller_id has no nulls
        Missing values: 0
[PASS] Sellers: seller_id is unique
        Duplicate rows for key: 0
[PASS] Orders: customer_id has no nulls
        Missing values: 0
[PASS] Order items: product_id has no nulls
   

In [33]:
# Add autometed tests for database integration

test_file = REPO_ROOT / "tests" / "test_pipeline.py"

print(test_file.read_text(encoding="utf-8"))


"""Tests for sales analytics pipeline outputs."""

import unittest
from pathlib import Path
import pandas as pd


REPO_ROOT = Path(__file__).resolve().parents[1]
PROCESSED_DIR = REPO_ROOT / "data" / "processed"


class TestPipelineOutputs(unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        cls.orders = pd.read_csv(
            PROCESSED_DIR / "order_analytics.csv"
        )
        cls.products = pd.read_csv(
            PROCESSED_DIR / "product_analytics.csv"
        )
        cls.categories = pd.read_csv(
            PROCESSED_DIR / "category_analytics.csv"
        )
        cls.customers = pd.read_csv(
            PROCESSED_DIR / "customer_analytics.csv"
        )
        cls.monthly = pd.read_csv(
            PROCESSED_DIR / "monthly_sales_analytics.csv"
        )
        cls.status = pd.read_csv(
            PROCESSED_DIR / "order_status_summary.csv"
        )

    def test_order_ids_are_unique(self):
        self.assertEqual(
            self.orders["order_id"]

In [38]:
# Append the new test cases

new_tests = r'''

class TestOptionalDatabaseIntegration(unittest.TestCase):

    def _run_mock_pipeline(self, database_url=None):
        import sys
        from unittest.mock import patch

        sys.path.insert(0, str(REPO_ROOT / "src"))
        from sales_pipeline import pipeline

        import tempfile

        with tempfile.TemporaryDirectory() as temp_dir:
            root = Path(temp_dir)
            raw_dir = root / "raw"
            output_dir = root / "processed"
            raw_dir.mkdir()
            output_dir.mkdir()

            def save_output(filename):
                def _save(*args, **kwargs):
                    path = output_dir / filename
                    path.write_text("test\n", encoding="utf-8")
                    return path
                return _save

            orders = pd.DataFrame({"order_id": ["a", "b"]})
            products = pd.DataFrame({"product_id": ["p"]})
            categories = pd.DataFrame({"category": ["c"]})
            customers = pd.DataFrame({"customer": ["u"]})
            monthly = pd.DataFrame({"month": ["2020-01"]})
            status = pd.DataFrame({"status": ["delivered"]})

            database_result = {
                "order_analytics": {
                    "source_rows": 2,
                    "loaded_rows": 2,
                }
            }

            with (
                patch.object(
                    pipeline.validate,
                    "validate_data",
                    return_value=[{"check": "mock"}],
                ),
                patch.object(
                    pipeline.validate,
                    "print_report",
                    return_value=True,
                ),
                patch.object(
                    pipeline.transform,
                    "build_order_analytics",
                    return_value=orders,
                ),
                patch.object(
                    pipeline.transform,
                    "save_order_analytics",
                    side_effect=save_output("order_analytics.csv"),
                ),
                patch.object(
                    pipeline.transform_products,
                    "build_product_analytics",
                    return_value=(products, categories),
                ),
                patch.object(
                    pipeline.transform_products,
                    "save_product_analytics",
                    side_effect=lambda *args: (
                        save_output("product_analytics.csv")(),
                        save_output("category_analytics.csv")(),
                    ),
                ),
                patch.object(
                    pipeline.transform_customers,
                    "build_customer_analytics",
                    return_value=customers,
                ),
                patch.object(
                    pipeline.transform_customers,
                    "save_customer_analytics",
                    side_effect=save_output("customer_analytics.csv"),
                ),
                patch.object(
                    pipeline.transform_sales,
                    "build_sales_analytics",
                    return_value=(monthly, status),
                ),
                patch.object(
                    pipeline.transform_sales,
                    "save_sales_analytics",
                    side_effect=lambda *args: (
                        save_output("monthly_sales_analytics.csv")(),
                        save_output("order_status_summary.csv")(),
                    ),
                ),
                patch.object(
                    pipeline.database,
                    "load_processed_csvs",
                    return_value=database_result,
                ) as database_loader,
            ):
                summary = pipeline.run_pipeline(
                    raw_dir,
                    output_dir,
                    database_url=database_url,
                )

                return summary, database_loader

    def test_csv_only_mode_skips_database_loader(self):
        summary, loader = self._run_mock_pipeline()

        self.assertFalse(summary["database_loaded"])
        self.assertIsNone(summary["database_results"])
        loader.assert_not_called()

    def test_database_mode_calls_loader_and_records_results(self):
        summary, loader = self._run_mock_pipeline(
            database_url="postgresql://test-placeholder"
        )

        self.assertTrue(summary["database_loaded"])
        self.assertIsNotNone(summary["database_results"])
        loader.assert_called_once()

        args = loader.call_args.kwargs
        self.assertEqual(
            args["database_url"],
            "postgresql://test-placeholder",
        )
        self.assertTrue(Path(args["processed_dir"]).is_dir())


'''

test_code = test_file.read_text(encoding="utf-8")

if "class TestOptionalDatabaseIntegration" in test_code:
    print("Integration tests already exist. No changes made.")
else:
    with test_file.open("a", encoding="utf-8") as file:
        file.write(new_tests)
    print("Added two optional database integration tests.")


Integration tests already exist. No changes made.


In [42]:

import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "unittest",
        "discover",
        "-s",
        str(REPO_ROOT / "tests"),
        "-p",
        "test_*.py",
        "-v",
    ],
    cwd=str(REPO_ROOT),
    capture_output=True,
    text=True,
)

print(result.stdout)
print(result.stderr)

print("Test exit code:", result.returncode)
assert result.returncode == 0, "Automated tests failed."




SALES ANALYTICS DATA PIPELINE

[1/5] Validating source data...

[2/5] Building order analytics...

[3/5] Building product and category analytics...

[4/5] Building customer analytics...

[5/5] Building sales performance analytics...

PIPELINE COMPLETED SUCCESSFULLY
Duration: 0.0 seconds
Validation checks: 1
Order rows: 2
Product rows: 1
Category rows: 1
Customer rows: 1
Monthly records: 1
Order status records: 1

Generated outputs:
- /tmp/tmpp8weeka0/processed/order_analytics.csv (5 bytes)
- /tmp/tmpp8weeka0/processed/product_analytics.csv (5 bytes)
- /tmp/tmpp8weeka0/processed/category_analytics.csv (5 bytes)
- /tmp/tmpp8weeka0/processed/customer_analytics.csv (5 bytes)
- /tmp/tmpp8weeka0/processed/monthly_sales_analytics.csv (5 bytes)
- /tmp/tmpp8weeka0/processed/order_status_summary.csv (5 bytes)
SALES ANALYTICS DATA PIPELINE

[1/5] Validating source data...

[2/5] Building order analytics...

[3/5] Building product and category analytics...

[4/5] Building customer analytics...

[5

Review Git Changes before commit

In [43]:

import subprocess

def run_git(*args):
    result = subprocess.run(
        ["git", *args],
        cwd=str(REPO_ROOT),
        capture_output=True,
        text=True,
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    print("Exit code:", result.returncode)
    return result

print("=== Git status ===")
run_git("status", "--short")

print("\n=== Diff summary ===")
run_git("diff", "--stat")

print("\n=== Changed tracked files ===")
run_git("diff", "--name-only")


=== Git status ===
 M 01_data_ingestion.ipynb
 M src/sales_pipeline/database.py
 M src/sales_pipeline/pipeline.py
 M tests/test_pipeline.py
?? Untitled0.ipynb

Exit code: 0

=== Diff summary ===
 01_data_ingestion.ipynb        |   2 +-
 src/sales_pipeline/database.py | 211 ++++++++++++++++++++++++++++++++++++++++-
 src/sales_pipeline/pipeline.py |  15 ++-
 tests/test_pipeline.py         | 136 ++++++++++++++++++++++++++
 4 files changed, 361 insertions(+), 3 deletions(-)

Exit code: 0

=== Changed tracked files ===
01_data_ingestion.ipynb
src/sales_pipeline/database.py
src/sales_pipeline/pipeline.py
tests/test_pipeline.py

Exit code: 0


CompletedProcess(args=['git', 'diff', '--name-only'], returncode=0, stdout='01_data_ingestion.ipynb\nsrc/sales_pipeline/database.py\nsrc/sales_pipeline/pipeline.py\ntests/test_pipeline.py\n', stderr='')

In [45]:

import subprocess

def git_check(*args):
    result = subprocess.run(
        ["git", *args],
        cwd=str(REPO_ROOT),
        capture_output=True,
        text=True,
    )
    print(result.stdout or "(no output)")
    if result.stderr:
        print(result.stderr)
    return result.returncode

print("=== Whitespace and patch checks ===")
git_check("diff", "--check")

print("\n=== Current requirements.txt ===")
requirements_file = REPO_ROOT / "requirements.txt"
print(requirements_file.read_text(encoding="utf-8"))

print("\n=== Pipeline changes ===")
git_check("diff", "--", "src/sales_pipeline/pipeline.py")

print("\n=== Notebook change summary ===")
git_check("diff", "--numstat", "--", "01_data_ingestion.ipynb")


=== Whitespace and patch checks ===
tests/test_pipeline.py:239: new blank line at EOF.


=== Current requirements.txt ===
pandas
numpy
psycopg2-binary
pytest


=== Pipeline changes ===
diff --git a/src/sales_pipeline/pipeline.py b/src/sales_pipeline/pipeline.py
index 36c0194..58b59f7 100644
--- a/src/sales_pipeline/pipeline.py
+++ b/src/sales_pipeline/pipeline.py
@@ -8,9 +8,10 @@ from . import transform
 from . import transform_products
 from . import transform_customers
 from . import transform_sales
+from . import database
 
 
-def run_pipeline(data_dir, output_dir):
+def run_pipeline(data_dir, output_dir, database_url=None):
     data_dir = Path(data_dir)
     output_dir = Path(output_dir)
     output_dir.mkdir(parents=True, exist_ok=True)
@@ -85,6 +86,16 @@ def run_pipeline(data_dir, output_dir):
                 f"Pipeline output is missing or empty: {path}"
             )
 
+    database_results = None
+
+    if database_url is not None:
+        print("\n[6/6] Loading analytics 

0

In [46]:
# Fix the dependencies and whitespace

# 1. Correct the PostgreSQL driver dependency.
requirements_file = REPO_ROOT / "requirements.txt"

requirements = [
    line.strip()
    for line in requirements_file.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

requirements = [
    line for line in requirements
    if line.lower() not in {"psycopg2-binary", "psycopg[binary]"}
]
requirements.append("psycopg[binary]")

requirements_file.write_text(
    "\n".join(requirements) + "\n",
    encoding="utf-8",
)

# 2. Remove extra trailing blank lines from the test file.
test_code = test_file.read_text(encoding="utf-8")
test_file.write_text(test_code.rstrip() + "\n", encoding="utf-8")

print("Updated requirements.txt:")
print(requirements_file.read_text(encoding="utf-8"))

print("Cleaned trailing whitespace in test_pipeline.py.")


Updated requirements.txt:
pandas
numpy
pytest
psycopg[binary]

Cleaned trailing whitespace in test_pipeline.py.


In [48]:
#varify
import subprocess

result = subprocess.run(
    ["git", "diff", "--check"],
    cwd=str(REPO_ROOT),
    capture_output=True,
    text=True,
)

print(result.stdout or "No whitespace errors.")
print(result.stderr)
print("Exit code:", result.returncode)

assert result.returncode == 0

No whitespace errors.

Exit code: 0


In [49]:
# Rerun 12 cases

import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "unittest",
        "discover",
        "-s",
        str(REPO_ROOT / "tests"),
        "-p",
        "test_*.py",
        "-v",
    ],
    cwd=str(REPO_ROOT),
    capture_output=True,
    text=True,
)

print(result.stdout)
print(result.stderr)
print("Test exit code:", result.returncode)

assert result.returncode == 0, "Automated tests failed."


SALES ANALYTICS DATA PIPELINE

[1/5] Validating source data...

[2/5] Building order analytics...

[3/5] Building product and category analytics...

[4/5] Building customer analytics...

[5/5] Building sales performance analytics...

PIPELINE COMPLETED SUCCESSFULLY
Duration: 0.0 seconds
Validation checks: 1
Order rows: 2
Product rows: 1
Category rows: 1
Customer rows: 1
Monthly records: 1
Order status records: 1

Generated outputs:
- /tmp/tmp83ddlukj/processed/order_analytics.csv (5 bytes)
- /tmp/tmp83ddlukj/processed/product_analytics.csv (5 bytes)
- /tmp/tmp83ddlukj/processed/category_analytics.csv (5 bytes)
- /tmp/tmp83ddlukj/processed/customer_analytics.csv (5 bytes)
- /tmp/tmp83ddlukj/processed/monthly_sales_analytics.csv (5 bytes)
- /tmp/tmp83ddlukj/processed/order_status_summary.csv (5 bytes)
SALES ANALYTICS DATA PIPELINE

[1/5] Validating source data...

[2/5] Building order analytics...

[3/5] Building product and category analytics...

[4/5] Building customer analytics...

[5

In [50]:
# Stage the intended files

import subprocess

files_to_stage = [
    "src/sales_pipeline/database.py",
    "src/sales_pipeline/pipeline.py",
    "tests/test_pipeline.py",
    "requirements.txt",
]

result = subprocess.run(
    ["git", "add", "--", *files_to_stage],
    cwd=str(REPO_ROOT),
    capture_output=True,
    text=True,
)

print(result.stdout or "Staging command completed.")
print(result.stderr)
print("Exit code:", result.returncode)

assert result.returncode == 0

print("\n--- Staged changes ---")
status = subprocess.run(
    ["git", "status", "--short"],
    cwd=str(REPO_ROOT),
    capture_output=True,
    text=True,
)
print(status.stdout)

print("\n--- Staged diff summary ---")
diff = subprocess.run(
    ["git", "diff", "--cached", "--stat"],
    cwd=str(REPO_ROOT),
    capture_output=True,
    text=True,
)
print(diff.stdout)


Staging command completed.

Exit code: 0

--- Staged changes ---
 M 01_data_ingestion.ipynb
M  requirements.txt
M  src/sales_pipeline/database.py
M  src/sales_pipeline/pipeline.py
M  tests/test_pipeline.py
?? 02_postgresql_data_warehouse.ipynb


--- Staged diff summary ---
 requirements.txt               |   2 +-
 src/sales_pipeline/database.py | 211 ++++++++++++++++++++++++++++++++++++++++-
 src/sales_pipeline/pipeline.py |  15 ++-
 tests/test_pipeline.py         | 134 ++++++++++++++++++++++++++
 4 files changed, 359 insertions(+), 3 deletions(-)



In [51]:
# Varify the Stage files
import subprocess

result = subprocess.run(
    ["git", "status", "--short"],
    cwd=str(REPO_ROOT),
    capture_output=True,
    text=True,
    check=True,
)

print(result.stdout)

result = subprocess.run(
    ["git", "diff", "--cached", "--check"],
    cwd=str(REPO_ROOT),
    capture_output=True,
    text=True,
)

print("Staged diff check:")
print(result.stdout or "No whitespace errors.")
assert result.returncode == 0

 M 01_data_ingestion.ipynb
M  requirements.txt
M  src/sales_pipeline/database.py
M  src/sales_pipeline/pipeline.py
M  tests/test_pipeline.py
?? 02_postgresql_data_warehouse.ipynb

Staged diff check:
No whitespace errors.


In [52]:
# Commit
import subprocess

result = subprocess.run(
    [
        "git", "commit", "-m",
        "Integrate Neon PostgreSQL data warehouse"
    ],
    cwd=str(REPO_ROOT),
    capture_output=True,
    text=True,
)

print(result.stdout)
print(result.stderr)
print("Exit code:", result.returncode)

assert result.returncode == 0, "Commit failed."

[main b825e5f] Integrate Neon PostgreSQL data warehouse
 4 files changed, 359 insertions(+), 3 deletions(-)


Exit code: 0


In [53]:
# Varify the repo and push to github

import subprocess

for args in [
    ["status", "--short", "--branch"],
    ["log", "-1", "--oneline"],
    ["rev-list", "--left-right", "--count", "origin/main...HEAD"],
]:
    result = subprocess.run(
        ["git", *args],
        cwd=str(REPO_ROOT),
        capture_output=True,
        text=True,
    )
    print(f"$ git {' '.join(args)}")
    print(result.stdout)
    if result.stderr:
        print(result.stderr)


$ git status --short --branch
## main...origin/main [ahead 3]
 M 01_data_ingestion.ipynb
?? 02_postgresql_data_warehouse.ipynb

$ git log -1 --oneline
b825e5f Integrate Neon PostgreSQL data warehouse

$ git rev-list --left-right --count origin/main...HEAD
0	3



In [55]:
#Push to github

import subprocess
from google.colab import userdata

github_token = userdata.get("GITHUB_TOKEN")

if not github_token:
    raise ValueError("GITHUB_TOKEN is missing from Colab Secrets.")

# Read the current remote URL without displaying credentials.
remote_result = subprocess.run(
    ["git", "remote", "get-url", "origin"],
    cwd=str(REPO_ROOT),
    capture_output=True,
    text=True,
    check=True,
)

remote_url = remote_result.stdout.strip()

if not remote_url.startswith("https://github.com/"):
    raise ValueError(
        "Expected an HTTPS GitHub remote. "
        "Inspect the remote configuration before continuing."
    )

# Supply the token only to this Git process through its environment.
# The configured remote URL remains unchanged.
askpass_script = (
    "#!/bin/sh\n"
    'case "$1" in\n'
    '  *Username*) printf "%s\\n" "x-access-token" ;;\n'
    '  *Password*) printf "%s\\n" "$GITHUB_TOKEN" ;;\n'
    '  *) exit 1 ;;\n'
    "esac\n"
)

import tempfile
import os

with tempfile.TemporaryDirectory() as temp_dir:
    askpass_path = os.path.join(temp_dir, "git-askpass.sh")

    with open(askpass_path, "w", encoding="utf-8") as file:
        file.write(askpass_script)

    os.chmod(askpass_path, 0o700)

    env = os.environ.copy()
    env["GITHUB_TOKEN"] = github_token
    env["GIT_ASKPASS"] = askpass_path
    env["GIT_TERMINAL_PROMPT"] = "0"

    result = subprocess.run(
        ["git", "push", "origin", "main"],
        cwd=str(REPO_ROOT),
        capture_output=True,
        text=True,
        env=env,
    )

print(result.stdout)
if result.returncode != 0:
    # Avoid printing any potentially sensitive credential material.
    print("Push failed. Git error output:")
    print(result.stderr.replace(github_token, "[REDACTED]"))

print("Exit code:", result.returncode)

if result.returncode != 0:
    raise RuntimeError("Git push failed; see the sanitized error above.")

print("GitHub push completed successfully.")



Exit code: 0
GitHub push completed successfully.


In [57]:

import subprocess

print("=== Notebook files in repository root ===")

for path in sorted(REPO_ROOT.glob("*.ipynb")):
    print(f"{path.name} | {path.stat().st_size:,} bytes")

print("\n=== Git status ===")

result = subprocess.run(
    ["git", "status", "--short", "--branch"],
    cwd=str(REPO_ROOT),
    capture_output=True,
    text=True,
)

print(result.stdout)


=== Notebook files in repository root ===
00_phase0_setup.ipynb | 46,651 bytes
01_data_ingestion.ipynb | 366,604 bytes
02_postgresql_data_warehouse.ipynb | 100,672 bytes

=== Git status ===
## main...origin/main
 M 01_data_ingestion.ipynb
?? 02_postgresql_data_warehouse.ipynb



In [59]:
import json
import subprocess

print("=== Change summary ===")
result = subprocess.run(
    ["git", "diff", "--stat", "--", "01_data_ingestion.ipynb"],
    cwd=str(REPO_ROOT),
    capture_output=True,
    text=True,
)
print(result.stdout or "No tracked changes detected.")

for name in ["01_data_ingestion.ipynb", "02_postgresql_data_warehouse.ipynb"]:
    path = REPO_ROOT / name
    notebook = json.loads(path.read_text(encoding="utf-8"))

    cells = notebook.get("cells", [])
    code_cells = [c for c in cells if c.get("cell_type") == "code"]
    output_cells = [c for c in code_cells if c.get("outputs")]

    print(f"\n{name}")
    print(f"Total cells: {len(cells)}")
    print(f"Code cells: {len(code_cells)}")
    print(f"Code cells with saved output: {len(output_cells)}")

=== Change summary ===
 01_data_ingestion.ipynb | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)


01_data_ingestion.ipynb
Total cells: 66
Code cells: 49
Code cells with saved output: 48

02_postgresql_data_warehouse.ipynb
Total cells: 44
Code cells: 37
Code cells with saved output: 33


In [60]:
import json
import re

patterns = {
    "Connection string": re.compile(
        r"postgres(?:ql)?://[^\s\"']+",
        re.IGNORECASE,
    ),
    "GitHub token": re.compile(
        r"\b(?:github_pat_[A-Za-z0-9_]+|gh[pousr]_[A-Za-z0-9_]+)\b"
    ),
}

for name in ["01_data_ingestion.ipynb", "02_postgresql_data_warehouse.ipynb"]:
    notebook = json.loads((REPO_ROOT / name).read_text(encoding="utf-8"))
    findings = []

    for i, cell in enumerate(notebook.get("cells", [])):
        text = "".join(cell.get("source", []))
        text += json.dumps(cell.get("outputs", []), ensure_ascii=False)

        for label, pattern in patterns.items():
            if pattern.search(text):
                findings.append((i, label))

    print(f"\n{name}:")
    if findings:
        for cell_no, label in findings:
            print(f"Review cell {cell_no}: {label}")
    else:
        print("No connection-string or GitHub-token pattern found.")


01_data_ingestion.ipynb:
No connection-string or GitHub-token pattern found.

02_postgresql_data_warehouse.ipynb:
Review cell 29: Connection string


In [61]:
import json
import re

path = REPO_ROOT / "02_postgresql_data_warehouse.ipynb"
nb = json.loads(path.read_text(encoding="utf-8"))

cell = nb["cells"][29]
source = "".join(cell.get("source", []))

# Redact PostgreSQL URLs
source = re.sub(
    r"postgres(?:ql)?://[^\s\"']+",
    "[REDACTED_POSTGRES_URL]",
    source,
    flags=re.IGNORECASE,
)

print("Cell 29 source (redacted):")
print(source[:3000])

Cell 29 source (redacted):
# Append the new test cases

new_tests = r'''

class TestOptionalDatabaseIntegration(unittest.TestCase):

    def _run_mock_pipeline(self, database_url=None):
        import sys
        from unittest.mock import patch

        sys.path.insert(0, str(REPO_ROOT / "src"))
        from sales_pipeline import pipeline

        import tempfile

        with tempfile.TemporaryDirectory() as temp_dir:
            root = Path(temp_dir)
            raw_dir = root / "raw"
            output_dir = root / "processed"
            raw_dir.mkdir()
            output_dir.mkdir()

            def save_output(filename):
                def _save(*args, **kwargs):
                    path = output_dir / filename
                    path.write_text("test\n", encoding="utf-8")
                    return path
                return _save

            orders = pd.DataFrame({"order_id": ["a", "b"]})
            products = pd.DataFrame({"product_id": ["p"]})
            categories = pd

In [62]:
import json
import re

path = REPO_ROOT / "02_postgresql_data_warehouse.ipynb"
nb = json.loads(path.read_text(encoding="utf-8"))

for i, cell in enumerate(nb.get("cells", [])):
    content = "".join(cell.get("source", []))
    for output in cell.get("outputs", []):
        content += json.dumps(output, ensure_ascii=False)

    for line_no, line in enumerate(content.splitlines(), start=1):
        if re.search(r"postgres(?:ql)?://", line, re.IGNORECASE):
            safe_line = re.sub(
                r"postgres(?:ql)?://[^\s\"']+",
                "[REDACTED_POSTGRES_URL]",
                line,
                flags=re.IGNORECASE,
            )
            print(f"Cell {i}, line {line_no}: {safe_line[:250]}")

Cell 29, line 124:             database_url="[REDACTED_POSTGRES_URL]"
Cell 29, line 134:             "[REDACTED_POSTGRES_URL]",
